# ⚽ Euro 2024 — Goalkeeper Analysis Pipeline

Full interactive version of the `scripts/` pipeline, in notebook form.

**Pipeline steps** (mirrors the standalone scripts):

| Step | Script | What it does |
|---|---|---|
| 1 | `clean_data.py` | Fetch Euro 2024 events from StatsBomb, extract GK stats |
| 2 | `zscore_analysis.py` | Z-score every GK across 7 metrics |
| 3 | `chart.py` | Tournament-wide charts (save %, xG scatter, actions, distribution) |
| 4 | `rank_goalkeepers.py` | Weighted composite ranking + Top 10 report |
| 5 | `dashboard.py` | Full stat-card + pitch-map dashboard per GK |
| 6 | `statistical_analysis.py` | Shot maps + GSaE (Goals Saved above Expectation) |

By default this notebook **loads the already-processed CSVs** in `data/` instead of re-fetching from the API (instant, no internet needed). Set `FETCH_FRESH = True` in the next cell to re-run the full fetch from scratch.

In [ ]:
import sys
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patheffects as pe
from matplotlib.gridspec import GridSpec
from pathlib import Path
from mplsoccer import Pitch, VerticalPitch

pd.set_option("display.max_columns", None)

# ── Paths ──────────────────────────────────────────────────────────────
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
FINAL_DIR = ROOT / "data" / "final"
CHARTS_DIR = ROOT / "reports" / "charts"
DASH_DIR = CHARTS_DIR / "dashboards"
IND_DIR = CHARTS_DIR / "individual"
for d in (RAW_DIR, PROC_DIR, FINAL_DIR, CHARTS_DIR, DASH_DIR, IND_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Config ─────────────────────────────────────────────────────────────
COMPETITION_ID = 55     # UEFA Euro
SEASON_ID = 282          # 2024
MIN_MATCHES = 2          # GKs below this are excluded from ranking
LONG_THRESHOLD = 32      # metres — UEFA "long ball" threshold

FETCH_FRESH = False   # set True to re-pull from the StatsBomb API instead of using saved CSVs

# ── Shared color palette (matches scripts/) ──────────────────────────────
BG = "#ffffff"
TEXT = "#1a1a1a"
BLUE = "#1a73e8"
GREEN = "#2e7d32"
RED = "#d32f2f"
YELLOW = "#f9a825"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": TEXT,
    "ytick.color": TEXT,
    "axes.edgecolor": "#cccccc",
    "grid.color": "#e0e0e0",
    "axes.titlecolor": TEXT,
})

print(f"ROOT: {ROOT}")
print(f"FETCH_FRESH: {FETCH_FRESH}")

## Step 1 · Load & Clean Data
*(`scripts/clean_data.py`)*

Pulls every Euro 2024 match's events from StatsBomb, then extracts three GK-relevant subsets: **shots faced**, **GK actions** (sweeper/claims/punches), and **goal kicks**.

In [ ]:
def safe_get(val, *keys, default=""):
    for k in keys:
        if isinstance(val, dict):
            val = val.get(k, default)
        else:
            return default
    return val


def flatten(series, *keys):
    return series.apply(lambda x: safe_get(x, *keys))


def fetch_and_clean():
    """Full clean_data.py pipeline: fetch from StatsBomb API and rebuild every CSV."""
    from statsbombpy import sb

    matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
    print(f"Euro 2024: {len(matches)} matches loaded")

    all_events = []
    for i, mid in enumerate(matches["match_id"]):
        try:
            ev = sb.events(match_id=mid)
            ev["match_id"] = mid
            all_events.append(ev)
        except Exception as e:
            print(f"  Failed match {mid}: {e}")
        if (i + 1) % 10 == 0 or (i + 1) == len(matches):
            print(f"  {i+1}/{len(matches)} loaded...")
    events = pd.concat(all_events, ignore_index=True)
    print(f"Total events: {len(events):,}")

    sample = events["type"].iloc[0] if "type" in events.columns else None
    is_nested = isinstance(sample, dict)

    if is_nested:
        events["type_name"] = flatten(events["type"], "name")
        events["player_name"] = flatten(events["player"], "name")
        events["player_id"] = flatten(events["player"], "id")
        events["team_name"] = flatten(events["team"], "name")
        pos_col = events["position"] if "position" in events.columns else pd.Series([{}] * len(events))
        events["position_name"] = flatten(pos_col, "name")
    else:
        rename = {"type": "type_name", "player": "player_name", "team": "team_name", "position": "position_name"}
        for old, new in rename.items():
            if old in events.columns and new not in events.columns:
                events[new] = events[old]

    # ── Shots ──
    shots_raw = events[events["type_name"] == "Shot"].copy()
    if is_nested:
        shots_raw["xg"] = shots_raw["shot"].apply(lambda x: safe_get(x, "statsbomb_xg", default=0) if isinstance(x, dict) else 0)
        shots_raw["outcome"] = shots_raw["shot"].apply(lambda x: safe_get(x, "outcome", "name"))
        shots_raw["gk_name"] = shots_raw["shot"].apply(lambda x: safe_get(x, "goalkeeper", "name"))
        shots_raw["shot_x"] = shots_raw["location"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
        shots_raw["shot_y"] = shots_raw["location"].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)
    else:
        xg_col = next((c for c in shots_raw.columns if "statsbomb_xg" in c), None)
        outcome_col = next((c for c in shots_raw.columns if c == "shot_outcome"), None)
        if xg_col:
            shots_raw["xg"] = shots_raw[xg_col]
        if outcome_col:
            shots_raw["outcome"] = shots_raw[outcome_col]
        if "location" in shots_raw.columns:
            shots_raw["shot_x"] = shots_raw["location"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
            shots_raw["shot_y"] = shots_raw["location"].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)

        gk_rows = events[events["position_name"] == "Goalkeeper"][["match_id", "team_name", "player_name"]].drop_duplicates()
        gk_map = {(r["match_id"], r["team_name"]): r["player_name"] for _, r in gk_rows.iterrows()}
        match_teams = events.groupby("match_id")["team_name"].unique()

        def find_opposing_gk(row):
            for t in match_teams.get(row["match_id"], []):
                if t != row["team_name"]:
                    return gk_map.get((row["match_id"], t), "")
            return ""

        shots_raw["gk_name"] = shots_raw.apply(find_opposing_gk, axis=1)

    shots_raw["xg"] = pd.to_numeric(shots_raw.get("xg", 0), errors="coerce").fillna(0)
    shots = shots_raw[shots_raw["gk_name"].notna() & (shots_raw["gk_name"] != "")].copy()

    # ── GK events ──
    gk_ev = events[events["type_name"] == "Goal Keeper"].copy()
    if is_nested:
        gk_ev["gk_action"] = gk_ev["goalkeeper"].apply(lambda x: safe_get(x, "type", "name") if isinstance(x, dict) else "")
        gk_ev["gk_outcome"] = gk_ev["goalkeeper"].apply(lambda x: safe_get(x, "outcome", "name") if isinstance(x, dict) else "")
    else:
        gk_ev["gk_action"] = gk_ev["goalkeeper_type"] if "goalkeeper_type" in gk_ev.columns else ""
        gk_ev["gk_outcome"] = gk_ev["goalkeeper_outcome"] if "goalkeeper_outcome" in gk_ev.columns else ""

    # ── Goal kicks ──
    passes_all = events[events["type_name"] == "Pass"].copy()
    if is_nested:
        passes_all["pass_type"] = passes_all["pass"].apply(lambda x: safe_get(x, "type", "name") if isinstance(x, dict) else "")
        passes_all["pass_length"] = passes_all["pass"].apply(lambda x: safe_get(x, "length", default=0) if isinstance(x, dict) else 0)
    else:
        passes_all["pass_type"] = passes_all["pass_type"] if "pass_type" in passes_all.columns else ""
        passes_all["pass_length"] = pd.to_numeric(passes_all["pass_length"], errors="coerce").fillna(0) if "pass_length" in passes_all.columns else 0
    goal_kicks = passes_all[passes_all["pass_type"] == "Goal Kick"].copy()

    # ── Aggregate GK table ──
    gk_shots = shots.groupby("gk_name").agg(
        shots_faced=("xg", "count"), xg_faced=("xg", "sum"),
        goals_conceded=("outcome", lambda x: (x == "Goal").sum()),
        shots_on_target=("outcome", lambda x: x.isin(["Saved", "Goal"]).sum()),
        saves=("outcome", lambda x: (x == "Saved").sum()),
        blocked_shots=("outcome", lambda x: (x == "Blocked").sum()),
    ).reset_index()
    gk_shots["save_pct"] = (gk_shots["saves"] / gk_shots["shots_on_target"].replace(0, np.nan) * 100).fillna(0)
    gk_shots["goals_prevented"] = gk_shots["xg_faced"] - gk_shots["goals_conceded"]

    gk_actions = gk_ev.groupby("player_name").agg(
        sweeper_actions=("gk_action", lambda x: (x == "Keeper Sweeper").sum()),
        claims=("gk_action", lambda x: (x == "Collected").sum()),   # StatsBomb uses 'Collected' not 'Claim'
        punches=("gk_action", lambda x: (x == "Punch").sum()),
        total_gk_events=("gk_action", "count"),
    ).reset_index().rename(columns={"player_name": "gk_name"})

    gk_gks = goal_kicks.groupby("player_name").agg(
        goal_kicks=("pass_type", "count"), avg_gk_length=("pass_length", "mean"),
    ).reset_index().rename(columns={"player_name": "gk_name"})

    gk_mp = gk_ev.groupby("player_name")["match_id"].nunique().reset_index()
    gk_mp.columns = ["gk_name", "matches_played"]

    gk_table = (gk_shots.merge(gk_actions, on="gk_name", how="left")
                .merge(gk_gks, on="gk_name", how="left")
                .merge(gk_mp, on="gk_name", how="left")
                .fillna(0))

    mp = gk_table["matches_played"].replace(0, 1)
    gk_table["sweeper_per_match"] = gk_table["sweeper_actions"] / mp
    gk_table["claims_per_match"] = gk_table["claims"] / mp
    gk_table["punches_per_match"] = gk_table["punches"] / mp
    gk_table["goal_kicks_per_match"] = gk_table["goal_kicks"] / mp
    gk_table["gp_per_match"] = gk_table["goals_prevented"] / mp
    gk_table["xg_per_match"] = gk_table["xg_faced"] / mp
    gk_table = gk_table[gk_table["matches_played"] >= MIN_MATCHES].copy()

    # ── Save everything ──
    raw_cols = ["match_id", "player_name", "team_name", "type_name", "gk_name", "xg", "outcome", "shot_x", "shot_y"]
    shots[[c for c in raw_cols if c in shots.columns]].to_csv(RAW_DIR / "euro2024_goalkeepers_raw.csv", index=False)
    gk_table.to_csv(PROC_DIR / "goalkeepers_clean.csv", index=False)
    shots.to_csv(PROC_DIR / "shots_faced.csv", index=False)
    gk_ev.to_csv(PROC_DIR / "gk_events.csv", index=False)
    goal_kicks.to_csv(PROC_DIR / "goal_kicks.csv", index=False)
    print(f"Saved {len(gk_table)} goalkeepers (>= {MIN_MATCHES} matches) to data/processed/")

    return gk_table, shots, gk_ev, goal_kicks

In [ ]:
if FETCH_FRESH:
    gk, shots, gk_ev, goal_kicks = fetch_and_clean()
else:
    gk = pd.read_csv(PROC_DIR / "goalkeepers_clean.csv")
    shots = pd.read_csv(PROC_DIR / "shots_faced.csv", low_memory=False)
    gk_ev = pd.read_csv(PROC_DIR / "gk_events.csv", low_memory=False)
    goal_kicks = pd.read_csv(PROC_DIR / "goal_kicks.csv", low_memory=False)
    print(f"Loaded from data/processed/: {len(gk)} goalkeepers, {len(shots)} shots, {len(gk_ev)} GK events, {len(goal_kicks)} goal kicks")

gk.head()

## Step 2 · Z-Score Analysis
*(`scripts/zscore_analysis.py`)*

Standardizes every GK across 7 metrics (mean 0, std 1), flipping sign for "lower is better" stats like goals conceded, so positive always means "better".

In [ ]:
METRICS = {
    "save_pct": {"label": "Save %", "higher_better": True},
    "goals_prevented": {"label": "Goals Prevented", "higher_better": True},
    "sweeper_per_match": {"label": "Sweeper/Match", "higher_better": True},
    "claims_per_match": {"label": "Claims/Match", "higher_better": True},
    "goals_conceded": {"label": "Goals Conceded", "higher_better": False},
    "xg_faced": {"label": "xG Faced", "higher_better": False},
    "gp_per_match": {"label": "Goals Prev./Match", "higher_better": True},
}


def compute_zscores(df):
    df = df.copy()
    for col, info in METRICS.items():
        mean, std = df[col].mean(), df[col].std()
        if std == 0:
            df[f"z_{col}"] = 0.0
        else:
            z = (df[col] - mean) / std
            df[f"z_{col}"] = z if info["higher_better"] else -z
    return df


gk = compute_zscores(gk)
z_cols = [f"z_{c}" for c in METRICS]
gk["z_avg"] = gk[z_cols].mean(axis=1)

gk.to_csv(FINAL_DIR / "goalkeeper_analysis.csv", index=False)
print(f"Saved: {FINAL_DIR / 'goalkeeper_analysis.csv'}")
gk[["gk_name", "z_avg"] + z_cols].sort_values("z_avg", ascending=False)

In [ ]:
z_cols_plot = [f"z_{c}" for c in METRICS]
labels = [METRICS[c]["label"] for c in METRICS]
gk_sorted = gk.sort_values("z_save_pct", ascending=False)
data = gk_sorted[z_cols_plot].values

fig, ax = plt.subplots(figsize=(14, max(8, len(gk) * 0.5)))
im = ax.imshow(data, cmap="RdYlGn", aspect="auto", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(gk_sorted)))
ax.set_yticklabels(gk_sorted["gk_name"].values, fontsize=9)

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        color = "black" if abs(val) < 1.2 else TEXT
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7.5, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Z-Score (positive = better)", color=TEXT)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT)
ax.set_title("Euro 2024 — Goalkeeper Z-Score Comparison", fontsize=14, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "zscore_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
gk_sorted = gk.sort_values("z_avg", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(7, len(gk) * 0.45)))
colors = [GREEN if v > 0 else RED for v in gk_sorted["z_avg"]]
ax.barh(gk_sorted["gk_name"], gk_sorted["z_avg"], color=colors, edgecolor="none", height=0.65)

for i, (val, name) in enumerate(zip(gk_sorted["z_avg"], gk_sorted["gk_name"])):
    offset = 0.02 if val >= 0 else -0.02
    ha = "left" if val >= 0 else "right"
    ax.text(val + offset, i, f"{val:.2f}", va="center", ha=ha, fontsize=9, color=TEXT)

ax.axvline(0, color="#8b949e", lw=1, ls="--", alpha=0.6)
ax.set_xlabel("Average Z-Score (higher = better)", fontsize=11)
ax.set_title("Euro 2024 — Overall GK Performance (Avg Z-Score)", fontsize=14, fontweight="bold", pad=14)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "zscore_overall.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 3 · Tournament Charts
*(`scripts/chart.py`)*

Four tournament-wide views: save % ranking, xG-vs-goals-conceded scatter, action volume per match, and goal-kick distribution.

In [ ]:
top = gk.sort_values("save_pct", ascending=True)
med = gk["save_pct"].median()

fig, ax = plt.subplots(figsize=(12, 7))
colors = [GREEN if v >= med else RED for v in top["save_pct"]]
bars = ax.barh(top["gk_name"], top["save_pct"], color=colors, edgecolor="none", height=0.65)
for bar, val in zip(bars, top["save_pct"]):
    ax.text(val + 0.4, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%",
            va="center", ha="left", fontsize=9.5, fontweight="bold", color=TEXT)
ax.axvline(med, color=YELLOW, ls="--", lw=1.5, alpha=0.8, label=f"Median: {med:.1f}%")
ax.set_xlabel("Save Percentage (%)", fontsize=11)
ax.set_title("Euro 2024 — Goalkeeper Save Percentage", fontsize=14, fontweight="bold", pad=14)
ax.set_xlim(0, 110)
ax.legend(framealpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "save_pct.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
sc = ax.scatter(
    gk["xg_faced"], gk["goals_conceded"], c=gk["goals_prevented"], cmap="RdYlGn",
    s=gk["matches_played"] * 35 + 40, edgecolors="#333333", linewidths=0.6, alpha=0.92,
)
lim = max(gk["xg_faced"].max(), gk["goals_conceded"].max()) + 0.5
ax.plot([0, lim], [0, lim], color="#8b949e", ls="--", lw=1.3, alpha=0.6, label="Expected (xG = GA)")
for _, row in gk.iterrows():
    ax.annotate(row["gk_name"].split()[-1], (row["xg_faced"], row["goals_conceded"]),
                xytext=(5, 4), textcoords="offset points", fontsize=7.5, color=TEXT, alpha=0.88)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label("Goals Prevented (xG - GA)", color=TEXT, fontsize=10)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT)
ax.set_xlabel("Total xG Faced", fontsize=11)
ax.set_ylabel("Goals Conceded", fontsize=11)
ax.set_title("Euro 2024 GKs — xG Faced vs Goals Conceded\n"
             "(Green = more goals prevented · bubble size = matches played)", fontsize=13, fontweight="bold", pad=14)
ax.legend(framealpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "xg_prevention.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
top10v = gk.nlargest(min(10, len(gk)), "total_gk_events").copy()
x = np.arange(len(top10v))
width = 0.6

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(top10v))
for col, color, label in [
    ("sweeper_per_match", BLUE, "Sweeper Actions / match"),
    ("claims_per_match", GREEN, "Claims / match"),
    ("punches_per_match", YELLOW, "Punches / match"),
]:
    vals = top10v[col].values
    ax.bar(x, vals, width, bottom=bottom, color=color, edgecolor="none", label=label)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels([n.split()[-1] for n in top10v["gk_name"]], rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Actions per Match", fontsize=11)
ax.set_title("Euro 2024 — Goalkeeper Actions per Match (Top 10 by Volume)", fontsize=13, fontweight="bold", pad=14)
ax.legend(framealpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "gk_actions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
gk_top = gk.nlargest(min(12, len(gk)), "goal_kicks_per_match")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(gk_top["gk_name"][::-1], gk_top["goal_kicks_per_match"][::-1], color=BLUE, edgecolor="none", height=0.65)
axes[0].set_xlabel("Goal Kicks per Match", fontsize=11)
axes[0].set_title("Goal Kicks per Match", fontsize=12, fontweight="bold")
axes[0].spines[["top", "right"]].set_visible(False)

sc = axes[1].scatter(gk["avg_gk_length"], gk["gp_per_match"], c=gk["save_pct"], cmap="YlGn",
                     s=80, edgecolors="#333333", lw=0.5)
for _, row in gk.iterrows():
    axes[1].annotate(row["gk_name"].split()[-1], (row["avg_gk_length"], row["gp_per_match"]),
                     xytext=(3, 3), textcoords="offset points", fontsize=7, color=TEXT, alpha=0.8)
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label("Save %", color=TEXT)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT)
axes[1].set_xlabel("Avg Goal Kick Length (m)", fontsize=11)
axes[1].set_ylabel("Goals Prevented per Match", fontsize=11)
axes[1].set_title("GK Length vs Goals Prevented/match", fontsize=12, fontweight="bold")
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("Euro 2024 — Goal Kick Distribution Analysis", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4 · Composite Ranking
*(`scripts/rank_goalkeepers.py`)*

Weighted blend of the Z-scores: **Goals Prevented 35% · Save% 30% · Sweeper 15% · Claims 10% · Goals Conceded 10%**.

In [ ]:
WEIGHTS = {
    "z_goals_prevented": 0.35,
    "z_save_pct": 0.30,
    "z_sweeper_per_match": 0.15,
    "z_claims_per_match": 0.10,
    "z_goals_conceded": 0.10,
}

gk["composite_score"] = sum(gk[col] * w for col, w in WEIGHTS.items())
gk = gk.sort_values("composite_score", ascending=False).reset_index(drop=True)
gk.index += 1
gk.index.name = "rank"

output_cols = [
    "gk_name", "matches_played", "shots_faced", "saves", "save_pct",
    "xg_faced", "goals_conceded", "goals_prevented",
    "sweeper_per_match", "claims_per_match",
    "z_save_pct", "z_goals_prevented", "z_sweeper_per_match",
    "z_claims_per_match", "z_goals_conceded", "z_gp_per_match", "z_xg_faced",
    "composite_score",
]
gk_rank = gk[[c for c in output_cols if c in gk.columns]]
gk_rank.to_csv(FINAL_DIR / "gk_rank.csv")
print(f"Saved: {FINAL_DIR / 'gk_rank.csv'}")

top10 = gk_rank.head(10).copy()
top10.to_csv(FINAL_DIR / "top10_gk_performance.csv")
print(f"Saved: {FINAL_DIR / 'top10_gk_performance.csv'}")

gk_rank

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
names = top10["gk_name"].values[::-1]
scores = top10["composite_score"].values[::-1]
pal = cm.viridis(np.linspace(0.3, 0.95, len(names)))

bars = ax.barh(names, scores, color=pal, edgecolor="none", height=0.65)
for bar, row_tuple in zip(bars, top10.iloc[::-1].itertuples()):
    w = bar.get_width()
    label = f"{row_tuple.composite_score:+.3f}  |  Sv% {row_tuple.save_pct:.0f}%  |  Prev {row_tuple.goals_prevented:+.1f}  |  MP {int(row_tuple.matches_played)}"
    if w >= 0:
        ax.text(w + 0.01, bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="left", fontsize=9, color=TEXT)
    else:
        ax.text(w - 0.01, bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="right", fontsize=9, color=TEXT)

ax.axvline(0, color="#8b949e", lw=1, ls="--", alpha=0.6)
ax.set_xlabel("Composite Z-Score (higher = better)", fontsize=11)
ax.set_title("Euro 2024 — Top 10 Goalkeeper Performance\n"
             "Score = Goals Prev. 35% · Save% 30% · Sweeper 15% · Claims 10% · GA 10%", fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(min(scores) - 0.15, max(scores) * 1.6)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "top10_composite.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
Z_METRICS = {
    "z_save_pct": "Save %", "z_goals_prevented": "Goals Prev.",
    "z_sweeper_per_match": "Sweeper", "z_claims_per_match": "Claims",
    "z_goals_conceded": "GA (inv.)", "z_gp_per_match": "GP/Match", "z_xg_faced": "xG Faced (inv.)",
}
z_cols_t = [c for c in Z_METRICS if c in top10.columns]
labels = [Z_METRICS[c] for c in z_cols_t]
data = top10[z_cols_t].values

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(data, cmap="RdYlGn", aspect="auto", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels([f"#{i+1} {n}" for i, n in enumerate(top10["gk_name"].values)], fontsize=9)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        color = "black" if abs(val) < 1.2 else TEXT
        ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=8, color=color)
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Z-Score (positive = better)", color=TEXT)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT)
ax.set_title("Euro 2024 — Top 10 GK Z-Score Breakdown", fontsize=14, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "top10_zscore_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def norm(series, ascending=True):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    n = (series - mn) / (mx - mn)
    return n if ascending else 1 - n

gk_r = top10.copy()
gk_r["n_save_pct"] = norm(gk_r["save_pct"])
gk_r["n_gp"] = norm(gk_r["goals_prevented"])
gk_r["n_sweeper"] = norm(gk_r["sweeper_per_match"])
gk_r["n_claims"] = norm(gk_r["claims_per_match"])
gk_r["n_ga"] = norm(gk_r["goals_conceded"], ascending=False)

metrics = ["n_save_pct", "n_gp", "n_sweeper", "n_claims", "n_ga"]
labels = ["Save %", "Goals\nPrevented", "Sweeper\nActions", "Claims", "GA\n(inv.)"]
N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
ax.set_facecolor(BG)
fig.patch.set_facecolor(BG)
for r in [0.2, 0.4, 0.6, 0.8, 1.0]:
    ax.plot(angles, [r] * (N + 1), color="#cccccc", lw=0.7, zorder=0)

cmap = plt.cm.tab10
top5 = gk_r.head(5)
for i, (_, row) in enumerate(top5.iterrows()):
    vals = [row[m] for m in metrics] + [row[metrics[0]]]
    color = cmap(i)
    ax.plot(angles, vals, "o-", lw=2.2, ms=5, color=color, label=row["gk_name"])
    ax.fill(angles, vals, alpha=0.18, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, color=TEXT, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticklabels([])
ax.spines["polar"].set_color("#cccccc")
ax.grid(False)
ax.set_title("Euro 2024 — Top 5 Goalkeepers (normalised 0-1)", fontsize=14, fontweight="bold", pad=35, color=TEXT)
ax.legend(loc="upper right", bbox_to_anchor=(1.38, 1.12), labelcolor=TEXT, facecolor=BG, edgecolor="#cccccc", fontsize=10)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "top10_radar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
CYAN = "#00897b"
GRAY = "#555555"

LONG_COLOR  = "#FF6D00"
SHORT_COLOR = "#1565C0"


def parse_loc(val):
    if isinstance(val, str):
        try:
            r = ast.literal_eval(val)
            if isinstance(r, (list, tuple)) and len(r) >= 2:
                return float(r[0]), float(r[1])
        except Exception:
            pass
    return None


def pitch_ax(fig, pos, title=""):
    ax = fig.add_subplot(pos)
    pitch = Pitch(pitch_type="statsbomb", pitch_color="grass",
                  line_color="white", linewidth=1, goal_type="box")
    pitch.draw(ax=ax)
    for sp in ax.spines.values():
        sp.set_color("#cccccc")
    if title:
        ax.set_title(title, fontsize=9, color=GRAY, pad=6)
    return pitch, ax


def panel_saves(fig, pos, gk_name, shots_df):
    saved  = shots_df[(shots_df["gk_name"] == gk_name) & (shots_df["outcome"] == "Saved")]
    goals  = shots_df[(shots_df["gk_name"] == gk_name) & (shots_df["outcome"] == "Goal")]
    others = shots_df[(shots_df["gk_name"] == gk_name) & (~shots_df["outcome"].isin(["Saved", "Goal"]))]
    pitch, ax = pitch_ax(fig, pos, "Saves  ●  vs Goals Conceded  ★")
    if not others.empty:
        pitch.scatter(others["shot_x"], others["shot_y"], s=18, c="#888888",
                      marker=".", alpha=0.5, ax=ax, zorder=3)
    if not saved.empty:
        pitch.scatter(saved["shot_x"], saved["shot_y"], s=55, c=CYAN, marker="o",
                      edgecolors="white", linewidths=0.4, alpha=0.85, ax=ax, zorder=5)
    if not goals.empty:
        pitch.scatter(goals["shot_x"], goals["shot_y"], s=80, c=RED, marker="*",
                      edgecolors="white", linewidths=0.4, alpha=0.9, ax=ax, zorder=6)


def panel_punches(fig, pos, gk_name, events_df):
    punches = events_df[(events_df["player_name"] == gk_name) & (events_df["gk_action"] == "Punch")]
    pitch, ax = pitch_ax(fig, pos, f"Punch Locations  ({len(punches)})")
    if punches.empty:
        ax.text(0.5, 0.5, "No punches recorded", transform=ax.transAxes,
                ha="center", va="center", fontsize=11, color="#aaaaaa", style="italic")
    for _, row in punches.iterrows():
        loc = parse_loc(row["location"])
        if loc:
            ax.plot(loc[0], loc[1], "*", color=CYAN, ms=11, mec="white", mew=0.3, alpha=0.85, zorder=5)


def panel_kicks(fig, pos, gk_name, kicks_df, long=True, threshold=32):
    gk_kicks = kicks_df[kicks_df["player_name"] == gk_name].copy()
    gk_kicks["pl"] = pd.to_numeric(gk_kicks["pass_length"], errors="coerce").fillna(0)
    subset = gk_kicks[gk_kicks["pl"] >= threshold] if long else gk_kicks[gk_kicks["pl"] < threshold]
    color = LONG_COLOR if long else SHORT_COLOR
    title = "Long Distribution  (≥32m)" if long else "Short Distribution  (<32m)"
    pitch, ax = pitch_ax(fig, pos, title)
    for _, row in subset.iterrows():
        s = parse_loc(row["location"])
        e = parse_loc(row["pass_end_location"])
        if s and e:
            ax.annotate("", xy=(e[0], e[1]), xytext=(s[0], s[1]),
                        arrowprops=dict(arrowstyle="-|>", color=color,
                                        alpha=0.75, lw=1.4, mutation_scale=6))
            ax.plot(s[0], s[1], "o", color=color, ms=3, alpha=0.85, zorder=5)
            ax.plot(e[0], e[1], "o", color=color, ms=2, alpha=0.6, zorder=4)


def panel_compare(fig, pos, gk_name, gk_df):
    row = gk_df[gk_df["gk_name"] == gk_name].iloc[0]
    metrics = ["save_pct", "goals_prevented", "sweeper_per_match", "claims_per_match"]
    labels  = ["Save %", "Goals\nPrevented", "Sweeper\n/Match", "Claims\n/Match"]

    gk_vals  = [row[m] for m in metrics]
    avg_vals = [gk_df[m].mean() for m in metrics]
    max_vals = [max(gk_df[m].max(), 0.01) for m in metrics]
    gk_n  = [v / mx for v, mx in zip(gk_vals,  max_vals)]
    avg_n = [v / mx for v, mx in zip(avg_vals, max_vals)]

    x = np.arange(len(metrics))
    w = 0.35

    ax = fig.add_subplot(pos)
    ax.set_facecolor(BG)
    ax.bar(x - w/2, gk_n,  w, color=CYAN,    label=gk_name.split()[0], edgecolor="none")
    ax.bar(x + w/2, avg_n, w, color="#cccccc", label="Tournament Avg",  edgecolor="none")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8, color=GRAY)
    ax.set_yticks([])
    ax.set_ylim(0, 1.25)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#cccccc")
    ax.legend(fontsize=7.5, framealpha=0, labelcolor=GRAY, loc="upper right")
    ax.set_title("vs Tournament Average", fontsize=9, color=GRAY, pad=6)


def panel_radar(fig, pos, gk_name, gk_stats):
    row = gk_stats[gk_stats["gk_name"] == gk_name]
    if row.empty:
        return
    row = row.iloc[0]
    metrics = {
        "save_pct":        ("Save %",        True),
        "saves":           ("Saves",          True),
        "shots_faced":     ("Shots\nFaced",   True),
        "goals_conceded":  ("Goals\nConc.",   False),
        "punches":         ("Punches",        True),
        "sweeper_actions": ("Sweeper",        True),
        "goal_kicks":      ("Long\nKicks",    True),
    }
    vals_raw, labels = [], []
    for col, (lbl, higher_better) in metrics.items():
        mean, std = gk_stats[col].mean(), gk_stats[col].std()
        z = 0.0 if std == 0 else (row[col] - mean) / std
        if not higher_better:
            z = -z
        vals_raw.append(z); labels.append(lbl)
    norm_vals = [max(0.02, min(0.98, (v + 3) / 6)) for v in vals_raw]
    N = len(labels)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    norm_vals += norm_vals[:1]; angles += angles[:1]

    ax = fig.add_subplot(pos, polar=True)
    ax.set_facecolor(BG)
    for r_val in [0.2, 0.4, 0.6, 0.8, 1.0]:
        ax.plot(angles, [r_val] * (N + 1), color="#cccccc", lw=0.5)
    ax.plot(angles, norm_vals, "o-", lw=2, ms=4.5, color=CYAN)
    ax.fill(angles, norm_vals, alpha=0.18, color=CYAN)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, color=GRAY, fontsize=8)
    ax.set_ylim(0, 1); ax.set_yticklabels([])
    ax.spines["polar"].set_color("#cccccc"); ax.grid(False)
    ax.set_title("Z-Score Profile", fontsize=10, color=GRAY, pad=18)


## Step 5 · Individual GK Dashboards
*(`scripts/dashboard.py`)*

A full stat-card + pitch-map dashboard for each of the Top 3 ranked goalkeepers: shot map, punch locations, long/short kick maps, and a Z-score radar.

In [ ]:
def generate_dashboard(gk_name, rank_num, gk_df, shots_df, events_df, kicks_df, rank_df):
    row = gk_df[gk_df["gk_name"] == gk_name]
    if row.empty:
        print(f"  Skipping {gk_name} — no stats")
        return

    rank_row = rank_df[rank_df["gk_name"] == gk_name]
    comp = rank_row.iloc[0]["composite_score"] if not rank_row.empty else 0.0

    fig = plt.figure(figsize=(19, 14), facecolor=BG)

    fig.text(0.5, 0.982, gk_name,
             ha="center", va="top", fontsize=20, fontweight="bold", color=TEXT)
    fig.text(0.025, 0.982, f"#{rank_num}",
             ha="left", va="top", fontsize=24, fontweight="bold", color=CYAN)
    fig.text(0.975, 0.982, f"Score: {comp:+.3f}",
             ha="right", va="top", fontsize=13, fontweight="bold", color=CYAN)

    gs = GridSpec(2, 3, figure=fig,
                  left=0.03, right=0.97, top=0.94, bottom=0.02,
                  wspace=0.12, hspace=0.18)

    panel_saves(fig, gs[0, 0], gk_name, shots_df)
    panel_punches(fig, gs[0, 1], gk_name, events_df)
    panel_kicks(fig, gs[0, 2], gk_name, kicks_df, long=True)
    panel_kicks(fig, gs[1, 0], gk_name, kicks_df, long=False)
    ax_blank = fig.add_subplot(gs[1, 1])
    ax_blank.set_facecolor(BG)
    ax_blank.axis("off")
    panel_radar(fig, gs[1, 2], gk_name, gk_df)

    fname = DASH_DIR / f"dashboard_{rank_num:02d}_{gk_name.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  Saved: {fname}")


top3 = gk_rank.head(3)
for i, (_, r) in enumerate(top3.iterrows(), 1):
    print(f"\n#{i} {r['gk_name']}")
    generate_dashboard(r["gk_name"], i, gk, shots, gk_ev, goal_kicks, gk_rank)


## Step 6 · Statistical Analysis — GSaE
*(`scripts/statistical_analysis.py`)*

GSaE = **Goals Saved above Expectation** (`xG faced − goals conceded`). Positive means the GK saved more than an average keeper would, given the quality of shots they faced.

In [ ]:
def plot_gsae_vs_save_pct(gk_df, highlight_gk=None):
    df = gk_df.copy()
    df["gsae"] = df["goals_prevented"]

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.scatter(df["gsae"], df["save_pct"], c=BLUE, s=70, alpha=0.9, edgecolors="white", linewidths=0.5, zorder=5)

    mask = df["gsae"].notna() & df["save_pct"].notna()
    if mask.sum() >= 2:
        z = np.polyfit(df.loc[mask, "gsae"], df.loc[mask, "save_pct"], 1)
        x_line = np.linspace(df["gsae"].min() - 0.3, df["gsae"].max() + 0.3, 100)
        ax.plot(x_line, np.polyval(z, x_line), color=BLUE, lw=2, alpha=0.5, zorder=3)

    if highlight_gk and highlight_gk in df["gk_name"].values:
        row = df[df["gk_name"] == highlight_gk].iloc[0]
        ax.scatter(row["gsae"], row["save_pct"], c="#ff6d00", s=200, marker="*", edgecolors="white", linewidths=1, zorder=10)
        ax.annotate(highlight_gk, (row["gsae"], row["save_pct"]), xytext=(8, 10), textcoords="offset points",
                    fontsize=9, fontweight="bold", color="#ff6d00", arrowprops=dict(arrowstyle="->", color="#ff6d00", lw=1.2))

    ax.axvline(0, color="#8b949e", ls=":", lw=0.8, alpha=0.5)
    ax.axhline(df["save_pct"].median(), color="#8b949e", ls=":", lw=0.8, alpha=0.5)
    ax.set_xlabel("GSaE (Goals Saved Above Expectation)", fontsize=11)
    ax.set_ylabel("Save %", fontsize=11)
    ax.set_title("GSaE vs. Save % Efficiency Analysis", fontsize=14, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    for _, row in df.iterrows():
        if highlight_gk and row["gk_name"] == highlight_gk:
            continue
        ax.annotate(row["gk_name"].split()[-1], (row["gsae"], row["save_pct"]), xytext=(4, 4),
                    textcoords="offset points", fontsize=6.5, color=TEXT, alpha=0.6)
    plt.tight_layout()
    suffix = f"_{highlight_gk.replace(' ', '_')}" if highlight_gk else "_all"
    plt.savefig(IND_DIR / f"gsae_efficiency{suffix}.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_gsae_bar_ranking(gk_df, highlight_gk=None):
    df = gk_df.copy()
    df["gsae"] = df["goals_prevented"]
    df = df.sort_values("gsae", ascending=True)

    colors = []
    for _, row in df.iterrows():
        if highlight_gk and row["gk_name"] == highlight_gk:
            colors.append("#ff6d00")
        elif row["gsae"] >= 0:
            colors.append(GREEN)
        else:
            colors.append(RED)

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(df["gk_name"], df["gsae"], color=colors, edgecolor="none", height=0.65)
    for bar, (_, row) in zip(bars, df.iterrows()):
        val = row["gsae"]
        offset = 0.05 if val >= 0 else -0.05
        ha = "left" if val >= 0 else "right"
        ax.text(val + offset, bar.get_y() + bar.get_height() / 2, f"{val:+.2f}", va="center", ha=ha, fontsize=8.5, fontweight="bold", color=TEXT)
    ax.axvline(0, color="#8b949e", ls="-", lw=1, alpha=0.6)
    ax.set_xlabel("GSaE (Goals Saved Above Expectation)", fontsize=11)
    ax.set_title("Euro 2024 — Goalkeeper GSaE Ranking\nGreen = overperforming xG · Red = underperforming", fontsize=13, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    suffix = f"_{highlight_gk.replace(' ', '_')}" if highlight_gk else "_all"
    plt.savefig(IND_DIR / f"gsae_ranking{suffix}.png", dpi=150, bbox_inches="tight")
    plt.show()


top_gk_name = gk_rank.iloc[0]["gk_name"]
print(f"Highlighting #1 ranked GK: {top_gk_name}")
plot_gsae_vs_save_pct(gk, highlight_gk=top_gk_name)
plot_gsae_bar_ranking(gk, highlight_gk=top_gk_name)

In [ ]:
def plot_shot_map_green(gk_name, shots_df):
    gk_shots = shots_df[shots_df["gk_name"] == gk_name].copy()
    if gk_shots.empty:
        print(f"No shot data for {gk_name}")
        return

    pitch = VerticalPitch(pitch_type="statsbomb", pitch_color="grass", line_color="white",
                          half=True, goal_type="box", linewidth=1.5, spot_scale=0.01)
    fig, ax = pitch.draw(figsize=(9, 11))
    fig.set_facecolor(BG)

    styles = {
        "Saved": dict(color="#e60000", marker="o", label="Saved", zorder=5),
        "Goal": dict(color="#530be4", marker="*", label="Goal Conceded", zorder=6),
        "Blocked": dict(color="#ffea00", marker="s", label="Blocked", zorder=4),
        "Off T": dict(color="#1971b8", marker="^", label="Off Target", zorder=3),
        "Post": dict(color="#ff9100", marker="D", label="Post/Bar", zorder=4),
        "Wayward": dict(color="#b0bec5", marker="v", label="Wayward", zorder=3),
    }
    for outcome, style in styles.items():
        mask = gk_shots["outcome"] == outcome
        if mask.sum() == 0:
            continue
        sizes = gk_shots.loc[mask, "xg"] * 800 + 60
        pitch.scatter(gk_shots.loc[mask, "shot_x"], gk_shots.loc[mask, "shot_y"], s=sizes,
                     c=style["color"], marker=style["marker"], alpha=0.88, edgecolors="white",
                     linewidths=0.6, label=f"{style['label']} ({mask.sum()})", ax=ax, zorder=style["zorder"])

    total = len(gk_shots)
    ga = (gk_shots["outcome"] == "Goal").sum()
    saves = (gk_shots["outcome"] == "Saved").sum()
    xg_tot = gk_shots["xg"].sum()
    sv_pct = saves / max((saves + ga), 1) * 100
    prev = xg_tot - ga

    fig.text(0.5, 0.97, gk_name, ha="center", va="top", fontsize=17, fontweight="bold", color=TEXT)
    fig.text(0.5, 0.935, "UEFA Euro 2024 — Shot Map (GK perspective)", ha="center", va="top", fontsize=10, color="#555555")
    fig.text(0.5, 0.905,
             f"Shots faced: {total}   |   Saves: {saves}   |   Goals: {ga}\n"
             f"Save %: {sv_pct:.1f}%   |   xG faced: {xg_tot:.2f}   |   Goals prevented: {prev:+.2f}",
             ha="center", va="top", fontsize=9, color=TEXT, linespacing=1.6)
    ax.legend(loc="upper left", fontsize=9, framealpha=0.7, facecolor=BG, edgecolor="#cccccc", labelcolor=TEXT)
    plt.tight_layout(rect=[0, 0.03, 1, 0.90])
    fname = IND_DIR / f"green_shot_map_{gk_name.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"  Saved: {fname}")


plot_shot_map_green(top_gk_name, shots)

## Done

All outputs are saved to:
- `data/final/` — `goalkeeper_analysis.csv`, `gk_rank.csv`, `top10_gk_performance.csv`
- `reports/charts/` — tournament charts + Top 10 charts
- `reports/charts/dashboards/` — individual dashboards (Top 3)
- `reports/charts/individual/` — GSaE + shot map for the #1 ranked GK

**Try it yourself:** change `top_gk_name = gk_rank.iloc[0]["gk_name"]` in Step 6 to any goalkeeper's name from `gk_rank["gk_name"]` and re-run the last two cells to get their individual shot map and GSaE highlight.